<!-- STEP X6 -->
# Optional extras (outside the timed hand-in run)

The extras below go beyond the brief. They are run separately from the main notebook, so the main
notebook stays within the five-minute limit. Three were chosen because they test the project's own
conclusions: **X6** (is the model only a fingerprint detector?), **X2** (does pseudo-labelling help with
far fewer labels?) and **X9** (how precise are the test-set numbers?).

In [1]:
# STANDIN A0
# Load the data, the split and the shared helpers (parts/00_setup.ipynb). Dropped from the hand-in
# notebook, where the setup cells come first.
%run 00_setup.ipynb

Python 3.13.7, scikit-learn 1.6.1, numpy 2.5.3, pandas 3.0.6
Working directory: /home/steven/Desktop/AI-for-Cybersecurity-Final-Mini-Project
Data folder: /home/steven/Desktop/AI-for-Cybersecurity-Final-Mini-Project/UNSW-NB15
score() and score_proba() ready


Rows with an infinite or missing value: 0
Loaded 447,915 flows x 76 features


Raw rows:                      447,915
Exact duplicates removed:      141,742
Conflicting rows removed:        1,068
Clean rows (all are used):     305,105


Dropped 9 constant columns: Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, URG Flag Count, CWR Flag Count, ECE Flag Count, Fwd Bytes/Bulk Avg, Fwd Packet/Bulk Avg, Fwd Bulk Rate Avg
67 features: 65 continuous, 2 0/1 columns (Fwd PSH Flags, RST Flag Count)
TUNE_IDX: 36,612 training rows for settings searches
Setup complete: 22 shared names ready in 29 s


In [2]:
# STANDIN B2
# The three Part B models with the settings chosen on validation in B1 and B2
# (parts/10_baseline_models.ipynb). Dropped from the hand-in notebook, where the real B1 and B2 cells come first.
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

BOOST_SETTINGS = {"learning_rate": 0.1, "max_leaf_nodes": 31, "random_state": SEED}
def make_boost():
    return HistGradientBoostingClassifier(**BOOST_SETTINGS)

tree = DecisionTreeClassifier(max_depth=12, random_state=SEED).fit(X_train, y_train)
logreg = Pipeline([("scale", StandardScaler()),
                   ("model", LogisticRegression(C=10.0, max_iter=2000, random_state=SEED))]).fit(X_train, y_train)
boost = make_boost().fit(X_train, y_train)
MODELS = {"tree": tree, "logreg": logreg, "boosting": boost}

In [3]:
# STANDIN E1
# add_noise from step E1 (parts/40_robustness.ipynb). Dropped from the hand-in notebook.
def add_noise(X, level, seed=SEED):
    noisy = X.copy()
    if level == 0:
        return noisy
    rng = np.random.default_rng(seed)
    values = noisy[CONT_COLS].to_numpy(dtype=float, copy=True)
    values += rng.standard_normal(values.shape) * (level * TRAIN_STD[CONT_COLS].to_numpy())
    noisy[CONT_COLS] = np.maximum(values, 0.0)
    return noisy

<!-- STEP X6 -->
## X6: the shortcut check, the black box without the attacker fingerprint

Parts D, E and G all found that the black box leans on the **TCP settings of the attacker machines**
(`Fwd Seg Size Min`, `FWD Init Win Bytes`, and on the victim side `Bwd Init Win Bytes`). Is it *only* a
fingerprint detector, or does it also learn the attacks' behaviour? We retrain gradient boosting
**without these three features**, with the same settings as B2 (not re-tuned), and compare it with the
original on four questions:

1. **How good is it on clean data** (validation and test)?
2. **How robust is it to measurement noise** (the E1 noise, levels 0.001 to 0.05)?
3. **Does it notice a new attack better** (the G1/G2 test: Fuzzers hidden from training)?
4. **What does it rely on instead** (its SHAP top 10, on the same 1,000 validation flows as D1)?

In [4]:
# STEP X6
TCP_FINGERPRINT = ["Fwd Seg Size Min", "FWD Init Win Bytes", "Bwd Init Win Bytes"]
NO_FP = [f for f in FEATURES if f not in TCP_FINGERPRINT]
boost_nofp = make_boost().fit(X_train[NO_FP], y_train)

# 1. Clean data.
x6_clean = pd.concat({
    part: pd.DataFrame({"with fingerprint (67 features)": score(boost, X, y),
                        "without fingerprint (64 features)": score(boost_nofp, X[NO_FP], y)}).T
    for part, X, y in (("validation", X_val, y_val), ("test", X_test, y_test))},
    names=["set", "model"])[["macro_f1", "recall", "pr_auc", "far"]]
x6_clean.to_csv(TABLES / "X6_clean.csv", float_format="%.4f")

# 2. Noise (same noise as E1, on the test set).
rows = []
for level in [0.0, 0.001, 0.005, 0.01, 0.02, 0.05]:
    X_noisy = add_noise(X_test, level)
    for name, model, cols in (("with fingerprint", boost, FEATURES), ("without fingerprint", boost_nofp, NO_FP)):
        rows.append({"level": level, "model": name, **score(model, X_noisy[cols], y_test)})
x6_noise = pd.DataFrame(rows)
x6_noise.to_csv(TABLES / "X6_noise.csv", index=False, float_format="%.4f")

print(f"Training without: {', '.join(TCP_FINGERPRINT)} ({len(NO_FP)} features left)")
print("\nRecall on the noisy test set:")
print(x6_noise.pivot(index="model", columns="level", values="recall").round(4).to_string())
x6_clean.round(4)

Training without: Fwd Seg Size Min, FWD Init Win Bytes, Bwd Init Win Bytes (64 features left)

Recall on the noisy test set:
level                 0.000   0.001   0.005   0.010   0.020   0.050
model                                                              
with fingerprint     0.9923  0.5128  0.4637  0.4604  0.4359  0.3035
without fingerprint  0.9910  0.8310  0.7892  0.7499  0.6409  0.4082


macro_f1  recall  pr_auc     far
set        model                                                              
validation with fingerprint (67 features)       0.9713  0.9924  0.9807  0.0282
           without fingerprint (64 features)    0.9711  0.9930  0.9806  0.0286
test       with fingerprint (67 features)       0.9718  0.9923  0.9814  0.0276
           without fingerprint (64 features)    0.9709  0.9910  0.9817  0.0281

In [5]:
# STEP X6
# 3. The hidden attack (G1/G2): Fuzzers removed from training, with and without the fingerprint.
HIDDEN = "Fuzzers"
keep = (type_train != HIDDEN).to_numpy()
is_hidden = (type_test == HIDDEN).to_numpy()
is_attack = (y_test == 1).to_numpy()

boost_hidden = make_boost().fit(X_train[keep], y_train[keep])
boost_nofp_hidden = make_boost().fit(X_train[keep][NO_FP], y_train[keep])
variants = [("with fingerprint", "all types", boost, FEATURES),
            ("with fingerprint", "Fuzzers hidden", boost_hidden, FEATURES),
            ("without fingerprint", "all types", boost_nofp, NO_FP),
            ("without fingerprint", "Fuzzers hidden", boost_nofp_hidden, NO_FP)]

rows = []
for name, trained_on, model, cols in variants:
    pred = model.predict(X_test[cols])
    rows.append({"model": name, "trained on": trained_on, "fuzzers_recall": pred[is_hidden].mean(),
                 "overall_recall": pred[is_attack].mean(), "far": float(pred[~is_attack].mean())})
x6_hidden = pd.DataFrame(rows)
x6_hidden.to_csv(TABLES / "X6_hidden.csv", index=False, float_format="%.4f")
x6_hidden.round(4)

,model,trained on,fuzzers_recall,overall_recall,far
0,with fingerprint,all types,0.9875,0.9923,0.0276
1,with fingerprint,Fuzzers hidden,0.6876,0.8880,0.0189
2,without fingerprint,all types,0.9864,0.9910,0.0281
3,without fingerprint,Fuzzers hidden,0.6870,0.8850,0.0189


In [6]:
# STEP X6
# 4. What the model without the fingerprint relies on: SHAP on the same 1,000 validation flows as D1.
import shap

GLOSSARY = pd.read_csv(TABLES / "T4_feature_glossary.csv", index_col="feature")
sample = X_val.sample(1000, random_state=SEED)
sv_nofp = shap.TreeExplainer(boost_nofp)(sample[NO_FP])
rank_nofp = pd.Series(np.abs(sv_nofp.values).mean(axis=0), index=NO_FP).sort_values(ascending=False)
x6_shap = pd.DataFrame({"share_of_shap_weight": (rank_nofp / rank_nofp.sum()).head(10)})
x6_shap = x6_shap.join(GLOSSARY[["meaning", "direction"]])
x6_shap.index.name = "feature"
x6_shap.to_csv(TABLES / "X6_shap_top10.csv", float_format="%.4f")
x6_shap.style.format({"share_of_shap_weight": "{:.1%}"})

,share_of_shap_weight,meaning,direction
feature,,,
Bwd Packets/s,46.1%,How fast the responder sent packets,backward
PSH Flag Count,16.0%,"Packets with the PSH flag (""deliver this data now"")",both
Bwd Header Length,6.2%,Total header bytes of the responder's packets,backward
Total Bwd packets,2.4%,Number of packets the responder (server/victim) sent back,backward
Packet Length Max,2.3%,"Largest packet payload in the conversation, either direction",both
Bwd Packet Length Max,1.7%,Largest packet payload the responder sent,backward
Fwd Packets/s,1.6%,How fast the initiator sent packets,forward
Fwd Packet Length Std,1.5%,How much the initiator's packet sizes vary,forward
FIN Flag Count,1.5%,"Packets with the FIN flag (""I am finished, close the connection"")",both


<!-- STEP X6 -->
### What the shortcut check shows

| Gradient boosting, test set | With fingerprint (67 features) | Without fingerprint (64 features) |
|---|---:|---:|
| Macro-F1, clean | 0.972 | 0.971 |
| Recall, clean | 0.992 | 0.991 |
| PR-AUC, clean | 0.981 | 0.982 |
| FAR, clean | 0.028 | 0.028 |
| Recall at noise level 0.001 | 0.513 | **0.831** |
| Recall at noise level 0.01 | 0.460 | **0.750** |
| Recall at noise level 0.05 | 0.304 | 0.408 |
| Recall on Fuzzers when Fuzzers are hidden from training (G2) | 0.688 | 0.687 |

**1. The fingerprint is not needed.** Without the three TCP-settings features the model is practically as
good on clean data: macro-F1 0.971 against 0.972, recall 0.991 against 0.992, validation the same. The
model used the fingerprint because it was the easiest signal, not because it was the only one: the
behaviour of the flows carries the same information.

**2. Without it, the model is much more robust to noise**, though still not robust: at noise level 0.001 it
catches 83% of the attacks instead of 51%, at 0.01 75% instead of 46%. Part of the fragility found in E3
came from the exact fingerprint values; the rest remains (exact zeros and narrow thresholds elsewhere).

**3. It does not recognise the new attack any better.** With Fuzzers hidden from training, recall on Fuzzers
is 0.687 with or without the fingerprint. The fingerprint was not what made the model miss fuzzing, and
removing it does not help it generalise to new kinds of attack; G3's labels are still needed.

**4. It now relies on what the victim does.** Its SHAP top 3 are `Bwd Packets/s` (how fast the other side
replies, 46% of the SHAP weight), `PSH Flag Count` (16%) and `Bwd Header Length` (the victim's header
bytes, 6%). These describe the conversation, mostly the victim's side of it, which the attacker cannot set
directly (E4: Fixed). And the cheap evasion of E4, copying the benign machines' TCP settings, no longer
applies at all, because the model does not look at those settings.

**Conclusion.** Dropping the attacker-fingerprint features costs almost nothing in accuracy, makes the
detector clearly more robust to measurement errors, and removes the cheapest way an attacker could evade it.
**For deployment we would train without them.** This also matters outside UNSW-NB15: on another network the
attackers would use other machines with other TCP settings, so a model that relies on behaviour has a much
better chance of carrying over than one that recognises a few lab machines.